# Notebook 02 — N-gram / HMM Baseline
**Project:** Sentiment Analysis Through the Ages  
**Tier:** 1 of 3 — Statistical Baseline  
**Dataset:** SST-2 (binary) and SST-5 (fine-grained)  

---

## What This Notebook Does

This notebook establishes the **numerical floor** for the project: the best performance achievable with classical NLP methods (N-gram TF-IDF features + Logistic Regression with L-BFGS optimisation + PoS tag features from a spaCy HMM-based tagger).

Every subsequent model (BiRNN, BERT) will be benchmarked against these numbers.

### Core concepts demonstrated here:
- **Tokenization** — whitespace/regex vs. spaCy
- **N-gram modelling** — unigram, bigram, trigram comparison
- **PoS Tagging / HMM** — spaCy's statistical tagger extracts syntactic features
- **TF-IDF** — converting text to weighted sparse vectors
- **Second-order optimisation (L-BFGS)** — vs. first-order (Adam, used later)
- **Evaluation** — accuracy, macro-F1, confusion matrix, per-class report

## 0. Setup

In [ ]:
# Install dependencies if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q datasets scikit-learn spacy seaborn
    !python -m spacy download en_core_web_sm -q

import os
os.makedirs('results', exist_ok=True)
print('Setup complete.')

In [ ]:
import re
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
import spacy

nlp = spacy.load('en_core_web_sm')
print('All imports successful.')

---
## 1. Load Data

In [ ]:
# ── Load SST-2 ────────────────────────────────────────────────────────────
sst2 = load_dataset('stanfordnlp/sst2')

train2_texts  = list(sst2['train']['sentence'])
train2_labels = list(sst2['train']['label'])
val2_texts    = list(sst2['validation']['sentence'])
val2_labels   = list(sst2['validation']['label'])

label_names_2 = ['negative', 'positive']

print('SST-2')
print(f'  Train : {len(train2_texts):,} sentences')
print(f'  Val   : {len(val2_texts):,} sentences')
print(f'  Sample: "{train2_texts[0]}" → {label_names_2[train2_labels[0]]}')

In [ ]:
# ── Load SST-5 ────────────────────────────────────────────────────────────
sst5 = load_dataset('SetFit/sst5')

train5_texts  = list(sst5['train']['text'])
train5_labels = list(sst5['train']['label'])
val5_texts    = list(sst5['validation']['text'])
val5_labels   = list(sst5['validation']['label'])

label_names_5 = ['very negative', 'negative', 'neutral', 'positive', 'very positive']

print('SST-5')
print(f'  Train : {len(train5_texts):,} sentences')
print(f'  Val   : {len(val5_texts):,} sentences')
print(f'  Sample: "{train5_texts[0]}" → {label_names_5[train5_labels[0]]}')

In [ ]:
# ── Class distribution plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, labels, names, title in [
    (axes[0], train2_labels, label_names_2, 'SST-2 Train Label Distribution'),
    (axes[1], train5_labels, label_names_5, 'SST-5 Train Label Distribution'),
]:
    counts = pd.Series(labels).value_counts().sort_index()
    ax.bar([names[i] for i in counts.index], counts.values, color=sns.color_palette('Blues_d', len(names)))
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/label_distribution.png', dpi=150)
plt.show()

---
## 2. Tokenisation

We use the simplest possible tokeniser here: lowercase + regex word extraction. This will be explicitly compared to spaCy (Tier 2) and WordPiece/BERT (Tier 3) in the final analysis.

In [ ]:
def simple_tokenize(text: str) -> list:
    """Lowercase + punctuation-stripping tokeniser."""
    return re.findall(r"[a-z']+", text.lower())

# Demonstrate tokenisation
examples = [
    "The film wasn't particularly good, but it wasn't terrible either.",
    "Absolutely brilliant — a masterpiece of modern cinema!",
    "I can't believe how boring this was.",
]
for ex in examples:
    print(f'Input : {ex}')
    print(f'Tokens: {simple_tokenize(ex)}')
    print()

---
## 3. PoS Feature Extraction (HMM / spaCy)

spaCy's tagger uses a statistical model (analogous to an HMM) to assign Universal PoS tags. We extract the *distribution* of PoS tags per sentence as an additional feature vector alongside TF-IDF. This adds syntactic signal — for example, sentiment-bearing sentences tend to have more adjectives (ADJ).

In [ ]:
UNIVERSAL_TAGS = [
    'ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET',
    'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN',
    'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X',
]
tag_to_idx = {tag: i for i, tag in enumerate(UNIVERSAL_TAGS)}

def extract_pos_features(texts: list, batch_size: int = 512) -> np.ndarray:
    n_tags = len(UNIVERSAL_TAGS)
    matrix = np.zeros((len(texts), n_tags), dtype=np.float32)
    for i, doc in enumerate(nlp.pipe(texts, batch_size=batch_size,
                                      disable=['ner', 'parser'])):
        counts = np.zeros(n_tags, dtype=np.float32)
        for token in doc:
            idx = tag_to_idx.get(token.pos_)
            if idx is not None:
                counts[idx] += 1
        total = counts.sum()
        if total > 0:
            counts /= total
        matrix[i] = counts
    return matrix

# Quick demo on 3 examples
demo_texts = [ex for ex in examples]
demo_pos = extract_pos_features(demo_texts)
demo_df = pd.DataFrame(demo_pos, columns=UNIVERSAL_TAGS)
demo_df.index = [f'ex{i+1}' for i in range(len(examples))]
print('PoS tag distributions (normalised):')
demo_df.round(3)

---
## 4. TF-IDF Vectoriser

In [ ]:
# Fit TF-IDF on SST-2 training set (unigrams + bigrams)
print('Fitting TF-IDF vectoriser...')
vectorizer = TfidfVectorizer(
    tokenizer=simple_tokenize,
    ngram_range=(1, 2),
    max_features=20_000,
    sublinear_tf=True,
    strip_accents='unicode',
    min_df=2,
)
vectorizer.fit(train2_texts)
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,} tokens')

# Sample top features by document frequency
feature_names = vectorizer.get_feature_names_out()
print(f'Sample features: {feature_names[:10].tolist()} ...')

---
## 5. Train Logistic Regression (L-BFGS — Second-Order Optimisation)

**Why L-BFGS?**  
Logistic regression minimises a convex cross-entropy loss. L-BFGS is a *quasi-Newton* (second-order) method that approximates the inverse Hessian to account for loss curvature. On convex problems, it converges in far fewer iterations than first-order methods like SGD or Adam, making it the natural choice for this baseline. In Tier 2 (BiRNN), we will switch to Adam and observe the trade-offs.

In [ ]:
# ── SST-2: build feature matrix ───────────────────────────────────────────
print('Transforming SST-2 training set...')
X_train2_tfidf = vectorizer.transform(train2_texts)
X_val2_tfidf   = vectorizer.transform(val2_texts)

print('Extracting PoS features for SST-2...')
pos_train2 = extract_pos_features(train2_texts)
pos_val2   = extract_pos_features(val2_texts)

X_train2 = hstack([X_train2_tfidf, csr_matrix(pos_train2)])
X_val2   = hstack([X_val2_tfidf,   csr_matrix(pos_val2)])
print(f'Feature matrix shape — train: {X_train2.shape}, val: {X_val2.shape}')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
clf2 = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    C=1.0,
    class_weight='balanced',
    n_jobs=-1,
)

print('Training on SST-2...')
t0 = time.time()
clf2.fit(X_train2, train2_labels)
train_time2 = time.time() - t0
print(f'Done. Training time: {train_time2:.1f}s  |  Iterations: {clf2.n_iter_[0]}')

---
## 6. Evaluate — SST-2

In [ ]:
y_pred2 = clf2.predict(X_val2)

acc2 = accuracy_score(val2_labels, y_pred2)
f1_2 = f1_score(val2_labels, y_pred2, average='macro')

print(f'SST-2 Validation Results')
print(f'  Accuracy  : {acc2:.4f}  ({acc2*100:.2f}%)')
print(f'  Macro-F1  : {f1_2:.4f}')
print()
print(classification_report(val2_labels, y_pred2, target_names=label_names_2))

In [ ]:
# Confusion matrix
cm2 = confusion_matrix(val2_labels, y_pred2)
plt.figure(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names_2, yticklabels=label_names_2)
plt.title('Confusion Matrix — N-gram Baseline (SST-2)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('results/confusion_baseline_sst2.png', dpi=150)
plt.show()

---
## 7. SST-5: Fine-Grained Classification

We now apply the **same** pipeline to SST-5. The drop in performance reveals the limits of bag-of-words representations for fine-grained distinctions.

In [ ]:
# Refit vectoriser on SST-5 training set
vec5 = TfidfVectorizer(
    tokenizer=simple_tokenize,
    ngram_range=(1, 2),
    max_features=20_000,
    sublinear_tf=True,
    strip_accents='unicode',
    min_df=2,
)
vec5.fit(train5_texts)

X_train5_tfidf = vec5.transform(train5_texts)
X_val5_tfidf   = vec5.transform(val5_texts)

print('Extracting PoS features for SST-5...')
pos_train5 = extract_pos_features(train5_texts)
pos_val5   = extract_pos_features(val5_texts)

X_train5 = hstack([X_train5_tfidf, csr_matrix(pos_train5)])
X_val5   = hstack([X_val5_tfidf,   csr_matrix(pos_val5)])
print(f'Feature matrix shape — train: {X_train5.shape}, val: {X_val5.shape}')

In [ ]:
clf5 = LogisticRegression(
    solver='lbfgs', max_iter=1000, C=1.0,
    multi_class='multinomial',
    class_weight='balanced', n_jobs=-1
)
t0 = time.time()
clf5.fit(X_train5, train5_labels)
print(f'Training time: {time.time()-t0:.1f}s  |  Iterations: {clf5.n_iter_[0]}')

y_pred5 = clf5.predict(X_val5)
acc5 = accuracy_score(val5_labels, y_pred5)
f1_5 = f1_score(val5_labels, y_pred5, average='macro')

print(f'\nSST-5 Validation Results')
print(f'  Accuracy  : {acc5:.4f}  ({acc5*100:.2f}%)')
print(f'  Macro-F1  : {f1_5:.4f}')
print()
print(classification_report(val5_labels, y_pred5, target_names=label_names_5))

In [ ]:
cm5 = confusion_matrix(val5_labels, y_pred5)
plt.figure(figsize=(7, 6))
sns.heatmap(cm5, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names_5, yticklabels=label_names_5)
plt.title('Confusion Matrix — N-gram Baseline (SST-5)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=20); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('results/confusion_baseline_sst5.png', dpi=150)
plt.show()

---
## 8. N-gram Ablation

Compare unigrams, unigrams+bigrams, and unigrams+bigrams+trigrams on SST-2. This illustrates the value of capturing multi-word sentiment phrases like *"not good"* or *"could have been better"*.

In [ ]:
ablation_rows = []
for ngram_range, label in [((1,1), 'unigram'), ((1,2), 'uni+bigram'), ((1,3), 'uni+bi+trigram')]:
    v = TfidfVectorizer(tokenizer=simple_tokenize, ngram_range=ngram_range,
                        max_features=20_000, sublinear_tf=True, min_df=2)
    v.fit(train2_texts)
    c = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0, n_jobs=-1)
    c.fit(v.transform(train2_texts), train2_labels)
    preds = c.predict(v.transform(val2_texts))
    row = {'ngram': label,
           'accuracy': round(accuracy_score(val2_labels, preds), 4),
           'macro_f1': round(f1_score(val2_labels, preds, average='macro'), 4)}
    ablation_rows.append(row)
    print(f"  {label:>15}  acc={row['accuracy']:.4f}  f1={row['macro_f1']:.4f}")

ablation_df = pd.DataFrame(ablation_rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = sns.color_palette('Blues_d', 3)
for ax, metric in zip(axes, ['accuracy', 'macro_f1']):
    ax.bar(ablation_df['ngram'], ablation_df[metric], color=colors)
    ax.set_title(f'N-gram Ablation — {metric} (SST-2)')
    ax.set_ylabel(metric)
    ymin = ablation_df[metric].min() - 0.02
    ymax = ablation_df[metric].max() + 0.02
    ax.set_ylim(ymin, ymax)
    for i, v in enumerate(ablation_df[metric]):
        ax.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('results/ngram_ablation.png', dpi=150)
plt.show()

---
## 9. Top Discriminative Features

Inspecting the highest-weight n-grams per class gives interpretability into what the model learned.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = clf2.coef_[0]  # binary: single coefficient vector

top_n = 20
top_pos_idx = np.argsort(coef)[-top_n:][::-1]
top_neg_idx = np.argsort(coef)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(feature_names[top_pos_idx][::-1], coef[top_pos_idx][::-1], color='#4C72B0')
axes[0].set_title('Top 20 Positive Features (SST-2 Baseline)')
axes[0].set_xlabel('Logistic Regression Weight')

axes[1].barh(feature_names[top_neg_idx], coef[top_neg_idx], color='#DD8452')
axes[1].set_title('Top 20 Negative Features (SST-2 Baseline)')
axes[1].set_xlabel('Logistic Regression Weight')

plt.tight_layout()
plt.savefig('results/top_features_sst2.png', dpi=150)
plt.show()

---
## 10. Summary & SST-2 → SST-5 Degradation Analysis

In [ ]:
summary = pd.DataFrame([
    {'Dataset': 'SST-2', 'Accuracy': acc2, 'Macro-F1': f1_2},
    {'Dataset': 'SST-5', 'Accuracy': acc5, 'Macro-F1': f1_5},
]).set_index('Dataset')

print('═'*45)
print('  BASELINE SUMMARY — N-gram + L-BFGS')
print('═'*45)
print(summary.round(4).to_string())
print()

acc_drop = acc2 - acc5
f1_drop  = f1_2 - f1_5
print(f'  SST-2 → SST-5 Accuracy drop : -{acc_drop:.4f} ({acc_drop*100:.1f} pp)')
print(f'  SST-2 → SST-5 Macro-F1 drop : -{f1_drop:.4f}  ({f1_drop*100:.1f} pp)')
print()
print('  This degradation is the core gap BiRNN (Tier 2) and')
print('  BERT (Tier 3) will attempt to close.')

# Save for comparison table
summary.to_csv('results/baseline_results.csv')
print('\nResults saved to results/baseline_results.csv')

---
## 11. Literature Benchmark Comparison

Position our results against published baselines from Socher et al. (2013).

In [ ]:
lit_benchmarks = pd.DataFrame([
    {'Model': 'NB (Socher 2013)',        'SST-2': 0.831, 'SST-5': 0.410},
    {'Model': 'SVM (Socher 2013)',       'SST-2': 0.794, 'SST-5': 0.407},
    {'Model': 'Our N-gram Baseline',     'SST-2': acc2,  'SST-5': acc5},
    {'Model': 'BiRNN + GloVe (target)',  'SST-2': 0.880, 'SST-5': 0.460},
    {'Model': 'BERT-base (target)',      'SST-2': 0.930, 'SST-5': 0.530},
]).set_index('Model')

print(lit_benchmarks.round(3).to_string())

ax = lit_benchmarks.plot(kind='bar', figsize=(11, 5), color=['#4C72B0', '#DD8452'])
ax.set_title('Accuracy Benchmarks: Baseline vs. Literature & Targets')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.35, 1.0)
ax.tick_params(axis='x', rotation=25)
ax.axhline(y=acc2, color='#4C72B0', linestyle='--', alpha=0.4, label='Our SST-2')
ax.axhline(y=acc5, color='#DD8452', linestyle='--', alpha=0.4, label='Our SST-5')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('results/benchmark_comparison.png', dpi=150)
plt.show()